In [8]:
!pip install streamlit pyngrok bcrypt pyjwt transformers textstat plotly -q
!pip install transformers==4.41.2 sentencepiece accelerate -q
!pip install streamlit transformers sentencepiece accelerate deep-translator

In [10]:
%%writefile app.py
import streamlit as st
from transformers import pipeline
from deep_translator import GoogleTranslator
import streamlit as st
import sqlite3
import bcrypt
import jwt
import datetime
import random
import smtplib
import re
from email.mime.text import MIMEText
from transformers import pipeline
import textstat
import plotly.express as px

# ---------------- CONFIG ----------------

SECRET_KEY = "textmorph_secret_key"

EMAIL_ID = "your_email@gmail.com"
EMAIL_PASSWORD = "your_app_password"

ADMIN_EMAIL = "admin@gmail.com"
ADMIN_PASSWORD = "admin123"

# ---------------- DATABASE ----------------

conn = sqlite3.connect("users.db", check_same_thread=False)
c = conn.cursor()

c.execute("""
CREATE TABLE IF NOT EXISTS users(
username TEXT,
email TEXT PRIMARY KEY,
password BLOB,
question TEXT,
answer TEXT,
failed_attempts INTEGER DEFAULT 0,
lock_until TEXT,
old_password BLOB
)
""")

conn.commit()

# ---------------- AI MODELS ----------------

@st.cache_resource
def load_models():

    summarizer = pipeline(
        "summarization",
        model="facebook/bart-large-cnn"
    )

    paraphraser = pipeline(
        "text2text-generation",
        model="Vamsi/T5_Paraphrase_Paws"
    )

    return summarizer, paraphraser

summarizer, paraphraser = load_models()

# ---------------- JWT ----------------

def create_token(email):

    payload = {
        "user": email,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
    }

    return jwt.encode(payload, SECRET_KEY, algorithm="HS256")

# ---------------- OTP ----------------

def send_otp(email):

    otp = str(random.randint(100000,999999))

    msg = MIMEText(f"Your OTP is {otp}")
    msg["Subject"] = "TextMorph OTP"
    msg["From"] = EMAIL_ID
    msg["To"] = email

    server = smtplib.SMTP("smtp.gmail.com",587)
    server.starttls()
    server.login(EMAIL_ID,EMAIL_PASSWORD)
    server.send_message(msg)
    server.quit()

    return otp

# ---------------- UI ----------------

st.set_page_config(page_title="TextMorph AI")

menu = ["Signup","Login","Forgot Password"]

choice = st.sidebar.selectbox("Menu",menu)

# ---------------- SIGNUP ----------------

if choice == "Signup":

    st.title("Signup")

    username = st.text_input("Username")
    email = st.text_input("Email")

    password = st.text_input("Password",type="password")
    confirm = st.text_input("Confirm Password",type="password")

    question = st.selectbox("Security Question",[
        "What is your pet name?",
        "What is your mother's maiden name?",
        "What is your favorite teacher?"
    ])

    answer = st.text_input("Answer")

    if st.button("Send OTP"):

        st.session_state["otp"] = send_otp(email)

        st.success("OTP sent")

    user_otp = st.text_input("Enter OTP")

    if st.button("Signup"):

        if password != confirm:
            st.error("Passwords do not match")

        elif user_otp != st.session_state.get("otp"):
            st.error("Invalid OTP")

        else:

            hashed = bcrypt.hashpw(password.encode(),bcrypt.gensalt())

            c.execute("INSERT INTO users VALUES(?,?,?,?,?,?,?,?)",
            (username,email,hashed,question,answer,0,None,None))

            conn.commit()

            st.success("Signup Successful")

# ---------------- LOGIN ----------------

elif choice == "Login":

    st.title("Login")

    email = st.text_input("Email")
    password = st.text_input("Password",type="password")

    if st.button("Login"):

        if email == ADMIN_EMAIL and password == ADMIN_PASSWORD:

            st.session_state["admin"] = True
            st.success("Admin Login")
            st.stop()

        c.execute("SELECT * FROM users WHERE email=?",(email,))
        user = c.fetchone()

        if not user:

            st.error("User not found")
            st.stop()

        failed_attempts = user[5]
        lock_until = user[6]

        if lock_until:

            lock_time = datetime.datetime.fromisoformat(lock_until)

            if datetime.datetime.now() < lock_time:

                st.error("Account locked 5 minutes")
                st.stop()

        if bcrypt.checkpw(password.encode(),user[2]):

            token = create_token(email)

            st.session_state["user"] = user[0]
            st.session_state["token"] = token

            st.success("Login Successful")

            c.execute("UPDATE users SET failed_attempts=0 WHERE email=?",(email,))
            conn.commit()

        else:

            failed_attempts += 1

            if failed_attempts >= 3:

                lock_time = datetime.datetime.now() + datetime.timedelta(minutes=5)

                c.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE email=?",
                (failed_attempts,lock_time.isoformat(),email))

            else:

                c.execute("UPDATE users SET failed_attempts=? WHERE email=?",
                (failed_attempts,email))

            conn.commit()

            st.error("Wrong Password")

# USER DASHBOARD

if "user" in st.session_state:

    st.title("TextMorph Dashboard")

    tool = st.sidebar.selectbox(
        "AI Tools",
        ["Summarizer","Paraphraser","Readability"]
    )

    # SUMMARIZER

    if tool == "Summarizer":

        st.header("Text Summarizer")

        text = st.text_area("Enter text")

        if st.button("Summarize"):

            result = summarizer(text,max_length=120,min_length=30)

            st.success(result[0]["summary_text"])

    # PARAPHRASER

    if tool == "Paraphraser":

        st.header("Paraphrase")

        text = st.text_area("Enter sentence")

        if st.button("Paraphrase"):

            result = paraphraser(
                "paraphrase: "+text,
                max_length=60
            )

            st.success(result[0]["generated_text"])

        st.subheader("Dataset Augmentation")

        sample = st.text_input("Enter sentence")

        if st.button("Augment Dataset"):

            aug = paraphraser(
                "paraphrase: "+sample,
                max_length=60
            )

            st.info(aug[0]["generated_text"])

    # READABILITY

    if tool == "Readability":

        st.header("Readability Dashboard")

        text = st.text_area("Enter text")

        if text:

            ease = textstat.flesch_reading_ease(text)
            grade = textstat.flesch_kincaid_grade(text)
            fog = textstat.gunning_fog(text)

            col1,col2,col3 = st.columns(3)

            col1.metric("Reading Ease",round(ease,2))
            col2.metric("Grade Level",round(grade,2))
            col3.metric("Fog Index",round(fog,2))

            data = {
                "Metric":["Reading Ease","Grade Level","Fog Index"],
                "Score":[ease,grade,fog]
            }

            fig = px.bar(data,x="Metric",y="Score")

            st.plotly_chart(fig)

    if st.button("Logout"):

        st.session_state.clear()
        st.rerun()

# ADMIN DASHBOARD

if "admin" in st.session_state:

    st.title("Admin Dashboard")

    c.execute("SELECT username,email FROM users")

    users = c.fetchall()

    st.table(users)

    if st.button("Logout Admin"):

        st.session_state.clear()
        st.rerun()
# PAGE TITLE
st.title("Multi-Language Convertor")

# LOAD MODELS
@st.cache_resource
def load_models():

    summarizer = pipeline(
        "summarization",
        model="sshleifer/distilbart-cnn-12-6"
    )

    paraphraser = pipeline(
        "text2text-generation",
        model="Vamsi/T5_Paraphrase_Paws"
    )

    return summarizer, paraphraser

summarizer, paraphraser = load_models()

# LANGUAGE SELECTOR
st.sidebar.title("Settings")

language = st.sidebar.selectbox(
    "Select Language",
    [
        "English",
        "Telugu",
        "Hindi",
        "Tamil",
        "Kannada",
        "Marathi",
        "French",
        "German"
    ]
)

# LANGUAGE CODES
lang_code = {
    "English":"en",
    "Telugu":"te",
    "Hindi":"hi",
    "Tamil":"ta",
    "Kannada":"kn",
    "Marathi":"mr",
    "French":"fr",
    "German":"de"
}

# TRANSLATION FUNCTION
def translate_text(text):

    if language != "English":

        translated = GoogleTranslator(
            source="auto",
            target=lang_code[language]
        ).translate(text)

        return translated

    return text


# USER INPUT
text = st.text_area("Enter your text")

tool = st.radio(
    "Choose Tool",
    ["Summarizer","Paraphraser"]
)

# PROCESS BUTTON
if st.button("Generate"):

    if text == "":
        st.warning("Please enter text")

    else:

        # SUMMARIZER
        if tool == "Summarizer":

            summary = summarizer(
                text,
                max_length=120,
                min_length=30,
                do_sample=False
            )

            result = summary[0]["summary_text"]

        # PARAPHRASER
        else:

            para = paraphraser(
                "paraphrase: " + text,
                max_length=120
            )

            result = para[0]["generated_text"]

        # TRANSLATE OUTPUT
        result = translate_text(result)

        st.subheader("Result")
        st.write(result)

# FOOTER
st.markdown("---")
st.caption("AI NLP Project • Summarization + Paraphrasing + Multi-Language")

Overwriting app.py


In [12]:
!ngrok config add-authtoken 3AbxG0gCiTPdX0Z6B03TdP6V5kq_eCLGFZwSiqBiWfFz9YmD

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!streamlit run app.py --server.port 8501 &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.245.204.16:8501

2026-03-07 11:07:58.226653: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-07 11:07:58.231832: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-07 11:07:58.247335: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772881678.272942   20489 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772881678.280682   20489 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS wh

In [14]:
from pyngrok import ngrok
print(ngrok.connect(8501))

NgrokTunnel: "https://angelita-managerial-westin.ngrok-free.dev" -> "http://localhost:8501"
